## TEST DATASET INITIALISATION

In [2]:
import sys
sys.path.append('../')

import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data_utils

import datasets
import importlib
importlib.reload(datasets)

from datasets.utils import dec2bin, dec2base
from datasets.random_hierarchy_model import sample_rules

ImportError: cannot import name 'dec2bin' from 'datasets.utils' (/home/2a/mb12724/miniconda3/envs/theory-lm/lib/python3.10/site-packages/datasets/utils/__init__.py)

In [ ]:
n = 32
v = 32
m = 32
L = 2
s = 2
input_size = s**L # number of pixels, actual input size is (input_size x num_features) because of one-hot encoding
num_data = n * (m**((s**L-1)//(s-1))) # total number of data
print(input_size, num_data)
seed_rules = 12345678 # seed of the random hierarchy model
rules = sample_rules( v, n, m, s, L, seed=seed_rules)
power = 2
zipf_prob=np.ones(m)
for i in range(m):
    zipf_prob[i]=(i+1)**(-1-power)
zipf_prob=torch.from_numpy(zipf_prob/np.sum(zipf_prob))
print(zipf_prob)
length = 32768
chosen_rule = torch.multinomial(zipf_prob, length, replacement=True)
for i in range(m):
    print(len([c for c in chosen_rule if c==i])/length)
train_size = 16
y = torch.randint(low=0, high=n, size=(train_size,))
print(y)
x = torch.clone(y)
for i in range(L):
    chosen_rule = torch.multinomial(zipf_prob, x.numel(), replacement=True).reshape(x.shape)
    print(chosen_rule.size())
#     chosen_rule = torch.randint(
#         low=0, high=rules[i].shape[1], size=x.shape
#     )
    print(x.shape, x.numel(), rules[i].shape[1], chosen_rule.size())
    x = rules[i][x, chosen_rule].flatten(start_dim=1)
    print(x.size())

4 1048576
tensor([8.3224e-01, 1.0403e-01, 3.0824e-02, 1.3004e-02, 6.6579e-03, 3.8529e-03,
        2.4263e-03, 1.6255e-03, 1.1416e-03, 8.3224e-04, 6.2527e-04, 4.8162e-04,
        3.7881e-04, 3.0329e-04, 2.4659e-04, 2.0318e-04, 1.6939e-04, 1.4270e-04,
        1.2133e-04, 1.0403e-04, 8.9864e-05, 7.8159e-05, 6.8401e-05, 6.0202e-05,
        5.3263e-05, 4.7351e-05, 4.2282e-05, 3.7912e-05, 3.4123e-05, 3.0824e-05,
        2.7936e-05, 2.5398e-05], dtype=torch.float64)


In [13]:
random.seed()
seed_rules = 12345678 # seed of the random hierarchy model
seed_sample = random.randrange(10000000,99999999)
print('sampling seed:', seed_sample)

v = 10
m = 10
n = 10
L = 2
s = 2
train_size = 262144 # size of the training set
test_size = 0 # size of the test set
input_format = 'long' # alternative: onehot
# to generate the full dataset: set trainset=num_data, test_size=0
bonus = False#dict.fromkeys(['tree', 'noise', 'synonyms', 'size'])
# bonus['size'] = 4

dataset = datasets.RandomHierarchyModel(
    num_features=v, # vocabulary size
    num_synonyms=m, # features multiplicity
    num_layers=L, # number of layers
    num_classes=n, # number of classes
    tuple_size=s, # number of branches of the tree
    seed_rules=seed_rules,
    zipf=2,
    layer=L,
    seed_sample=seed_sample,
    train_size=train_size,
    test_size=test_size,
    input_format=input_format,
    whitening=0, # 1 to whiten the input
    replacement=True,
    bonus=bonus
)

print(dir(dataset)) 
# for the input points call trainset.input
print(dataset.features.size()) # dimension: train_size x num_features x input_size
# for the labels call trainset.output
print(dataset.labels.size()) # dimension: train_size

sampling seed: 18808865
['__add__', '__annotations__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_is_protocol', 'features', 'labels', 'num_classes', 'num_features', 'num_layers', 'num_synonyms', 'prob', 'rules', 'transform', 'tuple_size']
torch.Size([262144, 4])
torch.Size([262144])


In [9]:
print(dataset.rules)
print(dataset.prob)

{0: tensor([[[9, 2],
         [1, 2],
         [8, 1],
         [7, 5],
         [2, 9],
         [3, 2],
         [7, 7],
         [6, 5],
         [9, 6],
         [9, 9]],

        [[0, 9],
         [4, 2],
         [1, 5],
         [1, 9],
         [4, 4],
         [0, 3],
         [6, 3],
         [0, 5],
         [4, 7],
         [2, 8]],

        [[6, 1],
         [6, 9],
         [8, 2],
         [5, 8],
         [8, 0],
         [7, 0],
         [1, 6],
         [9, 4],
         [0, 8],
         [9, 7]],

        [[0, 4],
         [5, 4],
         [3, 5],
         [3, 9],
         [4, 8],
         [2, 3],
         [0, 7],
         [7, 1],
         [8, 7],
         [1, 3]],

        [[5, 1],
         [3, 4],
         [2, 5],
         [3, 1],
         [1, 1],
         [3, 3],
         [3, 7],
         [9, 5],
         [7, 4],
         [8, 4]],

        [[6, 4],
         [3, 8],
         [1, 4],
         [8, 9],
         [0, 6],
         [9, 1],
         [4, 6],
         [5, 7],


In [10]:
print(dataset.features.size())

count = torch.zeros(m)
print(count)
total = 0.

for i in range(16384):

    if dataset.labels[i]==0:

        total += 1
        for j in range(m):

            if torch.equal(dataset.features[i]-1,dataset.rules[0][0][j]):
                count[j] += 1
print(count/total)
print(dataset.prob)

torch.Size([262144, 2])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor([0.8350, 0.0938, 0.0388, 0.0156, 0.0075, 0.0019, 0.0031, 0.0025, 0.0019,
        0.0000])
tensor([0.8351, 0.1044, 0.0309, 0.0130, 0.0067, 0.0039, 0.0024, 0.0016, 0.0011,
        0.0008], dtype=torch.float64)
